# Lab 9: Support Vector Machine (SVM) and Principal Component Analysis (PCA)

## Introduction

This lab explores two fundamental machine learning techniques:

**Support Vector Machine (SVM):** A powerful supervised learning algorithm used for classification and regression. SVM finds the optimal hyperplane that separates different classes in the feature space by maximizing the margin between them. It is particularly effective for high-dimensional data and can handle non-linear decision boundaries through kernel functions.

**Principal Component Analysis (PCA):** An unsupervised dimensionality reduction technique that transforms high-dimensional data into a lower-dimensional space while preserving as much variance as possible. PCA identifies orthogonal components (principal components) that capture the maximum variance in the data, enabling visualization, noise reduction, and computational efficiency.

In this lab, we will:
- Implement SVM on the Breast Cancer Wisconsin (Diagnostic) dataset
- Investigate the effect of SVM hyperparameter C
- Implement PCA on the Wine dataset
- Analyze principal components and explained variance
- Compare PCA with Linear Discriminant Analysis (LDA) as an extension

## Aim

To implement and evaluate Support Vector Machine (SVM) for classification and Principal Component Analysis (PCA) for dimensionality reduction, understanding their theoretical foundations, practical applications, and comparative performance.

## Objectives

### Part A: Support Vector Machine (SVM)
- Load and preprocess the Breast Cancer Wisconsin (Diagnostic) dataset
- Perform train-test split with stratification
- Standardize numerical features (critical for SVM)
- Train a Linear SVM classifier
- Evaluate using accuracy, precision, recall, F1-score, and confusion matrix
- Investigate the effect of hyperparameter C on model performance

### Part B: Principal Component Analysis (PCA)
- Load and explore the Wine dataset
- Standardize features before PCA
- Reduce 13 features to 2 principal components
- Calculate and interpret explained variance
- Determine minimum components for 95% variance retention
- Visualize transformed data in 2D
- Interpret PC1 and PC2 loadings
- Compare original vs transformed datasets
- Discuss advantages, limitations, and applications

### Extra Credit: Linear Discriminant Analysis (LDA)
- Apply LDA to the Wine dataset
- Reduce to two linear discriminants
- Compare LDA with PCA in terms of class separability

## Required Libraries

- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computations
- **matplotlib.pyplot**: Data visualization
- **seaborn**: Statistical data visualization
- **sklearn.preprocessing.StandardScaler**: Feature standardization
- **sklearn.model_selection.train_test_split**: Splitting data into training and testing sets
- **sklearn.svm.SVC**: Support Vector Machine classifier
- **sklearn.decomposition.PCA**: Principal Component Analysis
- **sklearn.discriminant_analysis.LinearDiscriminantAnalysis**: Linear Discriminant Analysis
- **sklearn.metrics**: Evaluation metrics (accuracy, precision, recall, F1, confusion matrix)

---

# PART A — SUPPORT VECTOR MACHINE (SVM)

## Dataset: Breast Cancer Wisconsin (Diagnostic)

### Dataset Description
The Breast Cancer Wisconsin (Diagnostic) dataset is a classic dataset for binary classification. It contains features computed from digitized images of fine needle aspirate (FNA) of a breast mass.

**Dataset Characteristics:**
- Number of samples: 569
- Number of features: 30 (computed from cell nuclei characteristics)
- Number of classes: 2 (Malignant, Benign)
- All features are numerical

**Feature Information:**
- ID: Unique identifier (to be excluded)
- Diagnosis: M = Malignant, B = Benign (target variable)
- 10 features computed for each cell nucleus: radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension
- The mean, standard error, and worst (mean of three largest) values are computed for each feature, resulting in 30 features

## A1. Dataset Loading and Understanding

### Purpose
Load the Breast Cancer Wisconsin dataset and understand its structure, features, and target distribution.

### Why This Step Is Needed
Understanding the dataset is crucial before implementing any classification algorithm. We need to identify the identifier column, target variable, feature types, and class distribution to make informed decisions about preprocessing and model selection.

In [ ]:
# Load the Breast Cancer Wisconsin dataset
# The dataset does not have headers, so we need to assign column names
column_names = ['ID', 'Diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
df_cancer = pd.read_csv('wdbc.data', names=column_names)

print("=" * 80)
print("BREAST CANCER WISCONSIN (DIAGNOSTIC) DATASET LOADED")
print("=" * 80)

In [ ]:
print("\n" + "=" * 80)
print("DATASET SHAPE")
print("=" * 80)
print(f"\nShape: {df_cancer.shape[0]} rows x {df_cancer.shape[1]} columns")
print(f"\nNumber of samples: {df_cancer.shape[0]}")
print(f"Number of features: {df_cancer.shape[1] - 2}")  # Excluding ID and Diagnosis
print(f"Number of classes: 2 (Malignant, Benign)")

In [ ]:
print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)
print(df_cancer.dtypes)

In [ ]:
print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)
print(df_cancer.isnull().sum())

In [ ]:
print("\n" + "=" * 80)
print("FIRST FIVE RECORDS")
print("=" * 80)
print(df_cancer.head())

In [ ]:
print("\n" + "=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)
class_counts = df_cancer['Diagnosis'].value_counts()
for label, count in class_counts.items():
    print(f"{label}: {count} samples ({count/len(df_cancer)*100:.1f}%)")

In [ ]:
# Visualize class distribution
plt.figure(figsize=(8, 6))
colors = ['#e74c3c', '#3498db']
bars = plt.bar(class_counts.index, class_counts.values, color=colors, 
               edgecolor='black', linewidth=1.5)
plt.xlabel('Diagnosis', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Class Distribution in Breast Cancer Dataset', fontsize=14, fontweight='bold', pad=15)
plt.grid(True, linestyle='--', alpha=0.5, axis='y')

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

## A1. Observations

**Observation:**
- The Breast Cancer Wisconsin dataset contains 569 samples with 30 features
- There are 2 target classes: M (Malignant) and B (Benign)
- Class distribution: 212 Malignant (37.3%), 357 Benign (62.7%) - moderately imbalanced
- All 30 features are numerical (float64)
- ID column is present and must be excluded from features
- Diagnosis is the target variable (categorical: M/B)
- No missing values in the dataset

**Interpretation:**
The dataset is moderately imbalanced with more benign cases than malignant cases. This imbalance is not severe but should be considered during evaluation. The absence of missing values simplifies preprocessing. All features are numerical, making them suitable for SVM after standardization.

**Practical Insight:**
The class imbalance (37% Malignant, 63% Benign) means that accuracy alone may not be a sufficient metric. We should also consider precision, recall, and F1-score to understand performance on each class. Stratification during train-test split will help preserve this distribution in both sets.

## A2. Preprocessing

### Purpose
Prepare the dataset for SVM by separating features and target, removing the identifier column, and encoding the target variable.

### Why This Step Is Needed
SVM requires numerical features and a numerical target. The identifier column provides no predictive information and should be removed. The categorical target (M/B) must be encoded to numerical values (0/1) for the classifier.

In [ ]:
# Drop the ID column (identifier, not a feature)
df_cancer = df_cancer.drop(columns=['ID'])

# Separate features (X) and target (y)
X_cancer = df_cancer.drop(columns=['Diagnosis'])
y_cancer = df_cancer['Diagnosis']

print("=" * 80)
print("FEATURES AND TARGET SEPARATED")
print("=" * 80)
print(f"\nFeatures shape: {X_cancer.shape}")
print(f"Target shape: {y_cancer.shape}")
print(f"\nNumber of features: {X_cancer.shape[1]}")
print(f"Feature columns: {list(X_cancer.columns[:5])}... (showing first 5)")

In [ ]:
# Encode target variable (M -> 1, B -> 0)
label_encoder = LabelEncoder()
y_cancer_encoded = label_encoder.fit_transform(y_cancer)

print("\n" + "=" * 80)
print("TARGET ENCODING")
print("=" * 80)
print(f"\nOriginal values: {list(label_encoder.classes_)}")
print(f"Encoded values: {list(range(len(label_encoder.classes_)))}")
print(f"\nM (Malignant) -> {label_encoder.transform(['M'])[0]}")
print(f"B (Benign) -> {label_encoder.transform(['B'])[0]}")

## A2. Observations

**Observation:**
- ID column successfully removed
- Features separated: 30 numerical features
- Target encoded: M (Malignant) → 1, B (Benign) → 0
- All features are numerical and ready for standardization

**Interpretation:**
The dataset is now ready for train-test split and feature scaling. The encoding maps Malignant to 1 and Benign to 0, which is a common convention where the positive class (the condition of interest) is encoded as 1.

**Practical Insight:**
Removing the ID column is critical because it provides no predictive information and could cause the model to memorize sample identifiers instead of learning meaningful patterns. The binary encoding of the target is necessary for SVM classifiers.

## A3. Train-Test Split

### Purpose
Split the dataset into training (80%) and testing (20%) sets with stratification to preserve class distribution.

### Why This Step Is Needed
Train-test split allows us to evaluate model performance on unseen data. Stratification ensures that the class distribution (37% Malignant, 63% Benign) is preserved in both training and testing sets, preventing biased evaluation due to class imbalance.

### train_test_split()

**Purpose**: Split arrays or matrices into random train and test subsets.

**Syntax**: `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`

**Parameters**:
- `X`: Features dataset
- `y`: Target variable
- `test_size`: Proportion of dataset for test split (0.0 to 1.0)
- `random_state`: Random seed for reproducibility
- `stratify`: If specified, data is split in a stratified fashion using this as the class labels

**Return value**: X_train, X_test, y_train, y_test (four arrays)

**Why it is appropriate here**: Stratified split ensures the class distribution is preserved in both training and testing sets, which is crucial for imbalanced datasets.

In [ ]:
# Perform 80:20 stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer_encoded, test_size=0.2, random_state=42, stratify=y_cancer_encoded
)

print("=" * 80)
print("TRAIN-TEST SPLIT (STRATIFIED)")
print("=" * 80)
print(f"\nTraining set: {X_train.shape[0]} samples ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

In [ ]:
# Verify class distribution in splits
print("\n" + "=" * 80)
print("CLASS DISTRIBUTION IN SPLITS")
print("=" * 80)

print("\nTraining set:")
for i, name in enumerate(label_encoder.classes_):
    count = (y_train == i).sum()
    print(f"  {name}: {count} samples ({count/len(y_train)*100:.1f}%)")

print("\nTesting set:")
for i, name in enumerate(label_encoder.classes_):
    count = (y_test == i).sum()
    print(f"  {name}: {count} samples ({count/len(y_test)*100:.1f}%)")

## A3. Observations

**Observation:**
- Training set has 455 samples (80%)
- Testing set has 114 samples (20%)
- Class distribution is preserved in both splits due to stratification:
  - Training: 37.4% Malignant, 62.6% Benign
  - Testing: 36.8% Malignant, 63.2% Benign
- Random state 42 ensures reproducibility

**Interpretation:**
The stratified split successfully preserved the class distribution in both training and testing sets. This is crucial for reliable model evaluation, especially with imbalanced datasets. The 80:20 ratio provides sufficient training data (455 samples) while retaining enough test data (114 samples) for meaningful evaluation.

**Practical Insight:**
Stratification is essential for imbalanced datasets. Without stratification, random splitting could accidentally create a test set with very different class proportions, leading to biased performance estimates. Stratification ensures that both the training and testing sets are representative of the overall population.

## A4. Feature Scaling

### Purpose
Standardize numerical features to have zero mean and unit variance before training SVM.

### Why This Step Is Needed
SVM is sensitive to feature scale. The algorithm finds the optimal hyperplane by maximizing the margin between classes. If features have different scales, features with larger numerical ranges can dominate the distance calculations, leading to poor model performance. Standardization ensures all features contribute equally to the decision boundary.

**Critical: Data Leakage Prevention**
The scaler must be fitted ONLY on the training data, then used to transform both training and testing data. Fitting on the entire dataset would leak information from the test set into the training process, leading to overly optimistic performance estimates.

### StandardScaler()

**Purpose**: Standardize features by removing the mean and scaling to unit variance.

**Syntax**: `StandardScaler().fit_transform(X_train)`

**Parameters**:
- `fit`: Compute mean and standard deviation for later scaling
- `transform`: Perform standardization using computed mean and std
- `fit_transform`: Fit to data, then transform it

**Return value**: Scaled array with mean=0 and std=1

**Why it is appropriate here**: StandardScaler is the standard method for feature scaling in sklearn. It ensures all features have the same scale, which is critical for SVM.

In [ ]:
# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data ONLY, then transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=" * 80)
print("FEATURE SCALING COMPLETED")
print("=" * 80)
print(f"\nTraining data shape after scaling: {X_train_scaled.shape}")
print(f"Testing data shape after scaling: {X_test_scaled.shape}")
print(f"\nTraining data mean (approx): {X_train_scaled.mean():.6f}")
print(f"Training data std (approx): {X_train_scaled.std():.6f}")

In [ ]:
# Display feature statistics before and after scaling
print("\n" + "=" * 80)
print("FEATURE STATISTICS BEFORE AND AFTER SCALING")
print("=" * 80)
print("\nBEFORE SCALING (first 5 features):")
print(X_train.iloc[:, :5].describe().loc[['mean', 'std']])

print("\nAFTER SCALING (first 5 features):")
print(pd.DataFrame(X_train_scaled[:, :5]).describe().loc[['mean', 'std']])

## A4. Observations

**Observation:**
- Features successfully standardized
- After scaling, training data has mean ≈ 0 and std ≈ 1
- Before scaling, features had vastly different scales (e.g., mean values ranging from ~14 to ~1000)
- After scaling, all features are on the same scale
- Scaler fitted only on training data, then applied to test data (no data leakage)

**Interpretation:**
Standardization successfully normalized all features to the same scale. This is critical for SVM because the algorithm uses distance-based calculations to find the optimal hyperplane. Without scaling, features with larger numerical ranges would dominate the decision boundary, leading to poor model performance.

**Practical Insight:**
Fitting the scaler only on training data and then using it to transform test data prevents data leakage. If we fitted on the entire dataset, information from the test set would influence the scaling parameters, leading to overly optimistic performance estimates. This is a critical best practice in machine learning.

## A5. Linear SVM Training

### Purpose
Train a Support Vector Machine classifier with a linear kernel on the standardized training data.

### Why This Step Is Needed
The linear kernel is the simplest SVM kernel and serves as a good baseline. It finds a linear decision boundary (hyperplane) that maximizes the margin between classes. For many datasets, a linear SVM provides excellent performance while being interpretable and computationally efficient.

### SVC()

**Purpose**: C-Support Vector Classification.

**Syntax**: `SVC(kernel='linear', C=1.0, random_state=42)`

**Parameters**:
- `kernel`: Specifies the kernel type to be used in the algorithm ('linear', 'poly', 'rbf', 'sigmoid')
- `C`: Regularization parameter. The strength of the regularization is inversely proportional to C. Must be strictly positive.
- `random_state`: Random seed for reproducibility

**Return value**: SVC classifier object

**Why it is appropriate here**: SVC is the sklearn implementation of Support Vector Machine. The linear kernel is specified as per the assignment requirements.

In [ ]:
# Train Linear SVM classifier
print("=" * 80)
print("TRAINING LINEAR SVM")
print("=" * 80)

svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_scaled, y_train)

print("\nTraining completed")
print(f"Number of support vectors: {len(svm_linear.support_vectors_)}")
print(f"Number of support vectors per class: {svm_linear.n_support_}")

In [ ]:
# Make predictions on test set
y_pred_svm = svm_linear.predict(X_test_scaled)

print("\n" + "=" * 80)
print("PREDICTIONS COMPLETED")
print("=" * 80)

## A6. Model Evaluation

### Purpose
Evaluate the Linear SVM classifier using accuracy, precision, recall, F1-score, and confusion matrix.

### Why This Step Is Needed
Multiple metrics provide a comprehensive view of model performance. Accuracy alone can be misleading, especially with imbalanced datasets. Precision, recall, and F1-score provide insights into performance on each class. The confusion matrix shows which classes are being confused with each other.

In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred_svm)
precision = precision_score(y_test, y_pred_svm)
recall = recall_score(y_test, y_pred_svm)
f1 = f1_score(y_test, y_pred_svm)

print("=" * 80)
print("MODEL EVALUATION METRICS")
print("=" * 80)
print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_svm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Linear SVM', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)
print(classification_report(y_test, y_pred_svm, target_names=label_encoder.classes_))

## A6. Model Evaluation: Observations

**Observation:**
- Linear SVM achieved high accuracy on the test set
- Confusion matrix shows the distribution of correct and incorrect predictions
- Classification report provides per-class metrics (precision, recall, F1-score)
- The model shows balanced performance across both classes

**Interpretation:**
The Linear SVM classifier demonstrates strong performance on the Breast Cancer dataset. The high accuracy indicates that the linear decision boundary effectively separates malignant and benign cases in the standardized feature space. The confusion matrix reveals the specific types of errors (false positives and false negatives), which is crucial for medical diagnosis applications.

**Practical Insight:**
In medical diagnosis, both precision and recall are important:
- High precision means fewer false positives (benign cases incorrectly classified as malignant), reducing unnecessary anxiety and procedures
- High recall means fewer false negatives (malignant cases incorrectly classified as benign), which is critical for early detection and treatment
- The F1-score provides a balanced measure of both precision and recall

## A7. SVM Concepts and Interpretation

### Hyperplane
A hyperplane is a decision boundary that separates different classes in the feature space. In a 2D space, it's a line; in 3D, it's a plane; in higher dimensions, it's a hyperplane. The goal of SVM is to find the optimal hyperplane that maximizes the margin between classes.

### Support Vectors
Support vectors are the data points that lie closest to the decision boundary (hyperplane). They are the most difficult points to classify and directly determine the position and orientation of the hyperplane. Removing non-support vectors does not affect the decision boundary.

### Margin
The margin is the distance between the hyperplane and the nearest data points (support vectors). SVM aims to maximize this margin, which leads to better generalization. A larger margin means the classifier is more confident and less likely to overfit.

### Hyperparameter C
C is a regularization parameter that controls the trade-off between:
- **Small C**: Allows more misclassifications (softer margin), which can lead to better generalization but underfitting
- **Large C**: Penalizes misclassifications heavily (harder margin), which can lead to overfitting but better training accuracy

### Kernel Trick
The kernel trick allows SVM to find non-linear decision boundaries by implicitly mapping the input features into a higher-dimensional space. Common kernels include:
- **Linear**: For linearly separable data
- **RBF (Radial Basis Function)**: For non-linear boundaries
- **Polynomial**: For polynomial decision boundaries

In this lab, we use a linear kernel as specified in the requirements.

## A8. Self-Learning: Effect of Hyperparameter C

### Purpose
Investigate how the SVM hyperparameter C affects model performance by training and evaluating SVMs with different C values.

### Why This Step Is Needed
Understanding the effect of hyperparameter C is crucial for model tuning. C controls the trade-off between maximizing the margin and minimizing classification error. By comparing performance across different C values, we can identify the optimal regularization strength for this dataset.

In [ ]:
# Test different C values
C_values = [0.01, 0.1, 1, 10, 100]
results = []

print("=" * 80)
print("INVESTIGATING EFFECT OF HYPERPARAMETER C")
print("=" * 80)

for C in C_values:
    svm = SVC(kernel='linear', C=C, random_state=42)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    n_sv = len(svm.support_vectors_)
    
    results.append({
        'C': C,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Support Vectors': n_sv
    })
    
    print(f"\nC = {C:6.2f}: Accuracy = {acc:.4f}, Precision = {prec:.4f}, Recall = {rec:.4f}, F1 = {f1:.4f}, SV = {n_sv}")

In [ ]:
# Create comparison table
results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("C VALUE COMPARISON TABLE")
print("=" * 80)
print(results_df.to_string(index=False))

In [ ]:
# Visualize effect of C on performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Effect of Hyperparameter C on SVM Performance', fontsize=16, fontweight='bold', y=0.995)

# Accuracy
axes[0, 0].plot(results_df['C'], results_df['Accuracy'], marker='o', color='#3498db', linewidth=2, markersize=8)
axes[0, 0].set_xscale('log')
axes[0, 0].set_xlabel('C (log scale)', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy vs C', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

# Precision
axes[0, 1].plot(results_df['C'], results_df['Precision'], marker='o', color='#2ecc71', linewidth=2, markersize=8)
axes[0, 1].set_xscale('log')
axes[0, 1].set_xlabel('C (log scale)', fontsize=12)
axes[0, 1].set_ylabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision vs C', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

# Recall
axes[1, 0].plot(results_df['C'], results_df['Recall'], marker='o', color='#f39c12', linewidth=2, markersize=8)
axes[1, 0].set_xscale('log')
axes[1, 0].set_xlabel('C (log scale)', fontsize=12)
axes[1, 0].set_ylabel('Recall', fontsize=12)
axes[1, 0].set_title('Recall vs C', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

# Support Vectors
axes[1, 1].plot(results_df['C'], results_df['Support Vectors'], marker='o', color='#e74c3c', linewidth=2, markersize=8)
axes[1, 1].set_xscale('log')
axes[1, 1].set_xlabel('C (log scale)', fontsize=12)
axes[1, 1].set_ylabel('Number of Support Vectors', fontsize=12)
axes[1, 1].set_title('Support Vectors vs C', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## A8. Self-Learning: Observations

**Observation:**
- Different C values produce different model performance
- As C increases, the number of support vectors typically decreases
- There is an optimal range for C where performance is best
- Very small C (0.01) may underfit, while very large C (100) may overfit

**Interpretation:**
The hyperparameter C controls the trade-off between margin maximization and classification error minimization:
- Small C values (0.01, 0.1) allow more misclassifications, resulting in a wider margin and potentially underfitting
- Large C values (10, 100) penalize misclassifications heavily, resulting in a narrower margin and potentially overfitting
- The optimal C value depends on the dataset and should be selected through cross-validation

**Practical Insight:**
For the Breast Cancer dataset, the performance is relatively stable across different C values, indicating that the data is well-suited for linear SVM. The number of support vectors decreases as C increases because the model becomes more strict about misclassifications. In practice, C should be tuned using cross-validation to find the optimal value for a given dataset.

---

# PART B — PRINCIPAL COMPONENT ANALYSIS (PCA)

## Dataset: Wine

### Dataset Description
The Wine dataset is a classic dataset for multiclass classification and dimensionality reduction. It contains the results of a chemical analysis of wines grown in the same region in Italy by three different cultivators.

**Dataset Characteristics:**
- Number of samples: 178
- Number of features: 13 (chemical constituents)
- Number of classes: 3 (cultivars)
- All features are numerical

**Feature Information:**
- Class: Cultivar (1, 2, 3) - target variable
- Alcohol: Alcohol content
- Malic acid: Malic acid content
- Ash: Ash content
- Alcalinity of ash: Alcalinity of ash
- Magnesium: Magnesium content
- Total phenols: Total phenols
- Flavanoids: Flavanoids content
- Nonflavanoid phenols: Nonflavanoid phenols
- Proanthocyanins: Proanthocyanins content
- Color intensity: Color intensity
- Hue: Hue
- OD280/OD315 of diluted wines: OD280/OD315 ratio
- Proline: Proline content

## B1. Dataset Loading and Understanding

### Purpose
Load the Wine dataset and understand its structure, features, and target distribution.

### Why This Step Is Needed
Understanding the dataset is crucial before implementing PCA. We need to know the number of samples, features, classes, and their distributions to interpret the PCA results correctly.

In [ ]:
# Load the Wine dataset
# The dataset does not have headers, so we need to assign column names
column_names_wine = ['Class', 'Alcohol', 'Malic_acid', 'Ash', 'Alcalinity_of_ash', 
                     'Magnesium', 'Total_phenols', 'Flavanoids', 'Nonflavanoid_phenols',
                     'Proanthocyanins', 'Color_intensity', 'Hue', 'OD280_OD315', 'Proline']
df_wine = pd.read_csv('wine.data', names=column_names_wine)

print("=" * 80)
print("WINE DATASET LOADED")
print("=" * 80)

In [ ]:
print("\n" + "=" * 80)
print("DATASET SHAPE")
print("=" * 80)
print(f"\nShape: {df_wine.shape[0]} rows x {df_wine.shape[1]} columns")
print(f"\nNumber of samples: {df_wine.shape[0]}")
print(f"Number of features: {df_wine.shape[1] - 1}")  # Excluding Class
print(f"Number of classes: 3 (Cultivars 1, 2, 3)")

In [ ]:
print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)
print(df_wine.dtypes)

In [ ]:
print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)
print(df_wine.isnull().sum())

In [ ]:
print("\n" + "=" * 80)
print("FIRST FIVE RECORDS")
print("=" * 80)
print(df_wine.head())

In [ ]:
print("\n" + "=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)
class_counts_wine = df_wine['Class'].value_counts().sort_index()
for label, count in class_counts_wine.items():
    print(f"Class {label}: {count} samples ({count/len(df_wine)*100:.1f}%)")

In [ ]:
# Visualize class distribution
plt.figure(figsize=(8, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = plt.bar(class_counts_wine.index, class_counts_wine.values, color=colors, 
               edgecolor='black', linewidth=1.5)
plt.xlabel('Class (Cultivar)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Class Distribution in Wine Dataset', fontsize=14, fontweight='bold', pad=15)
plt.grid(True, linestyle='--', alpha=0.5, axis='y')

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

## B1. Observations

**Observation:**
- The Wine dataset contains 178 samples with 13 features
- There are 3 target classes (cultivars): Class 1, 2, 3
- Class distribution: Class 1 (59 samples, 33.1%), Class 2 (71 samples, 39.9%), Class 3 (48 samples, 26.9%) - relatively balanced
- All 13 features are numerical (float64)
- Class is the target variable (categorical: 1, 2, 3)
- No missing values in the dataset

**Interpretation:**
The Wine dataset is relatively balanced across the three classes, which is ideal for PCA visualization. The absence of missing values simplifies preprocessing. All features are numerical and represent different chemical properties of wine, making them suitable for PCA.

**Practical Insight:**
The balanced class distribution ensures that PCA visualization will not be biased toward any particular class. The 13 features represent different chemical measurements that may be correlated, which is exactly the scenario where PCA is most beneficial—it can identify the underlying patterns and reduce dimensionality while preserving the most important information.

## B2. Preprocessing for PCA

### Purpose
Prepare the Wine dataset for PCA by separating features and target, and standardizing the features.

### Why This Step Is Needed
PCA is sensitive to feature scale. Features with larger numerical ranges will dominate the variance calculation, leading to biased principal components. Standardization ensures all features contribute equally to the PCA. Additionally, PCA is an unsupervised technique, so we separate the target (class) from features before transformation.

In [ ]:
# Separate features (X) and target (y)
X_wine = df_wine.drop(columns=['Class'])
y_wine = df_wine['Class']

print("=" * 80)
print("FEATURES AND TARGET SEPARATED")
print("=" * 80)
print(f"\nFeatures shape: {X_wine.shape}")
print(f"Target shape: {y_wine.shape}")
print(f"\nNumber of features: {X_wine.shape[1]}")
print(f"Feature columns: {list(X_wine.columns)}")

In [ ]:
# Standardize features before PCA
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

print("\n" + "=" * 80)
print("FEATURE STANDARDIZATION COMPLETED")
print("=" * 80)
print(f"\nScaled data shape: {X_wine_scaled.shape}")
print(f"Mean (approx): {X_wine_scaled.mean():.6f}")
print(f"Std (approx): {X_wine_scaled.std():.6f}")

## B2. Observations

**Observation:**
- Features and target successfully separated
- 13 features standardized to have mean ≈ 0 and std ≈ 1
- All features now on the same scale, ready for PCA

**Interpretation:**
Standardization is critical for PCA because PCA maximizes variance. Without standardization, features with larger scales would dominate the principal components regardless of their actual importance. After standardization, all features contribute equally to the variance calculation.

**Practical Insight:**
For the Wine dataset, features like Proline (range ~500-1680) and Alcohol (range ~11-15) have vastly different scales. Without standardization, Proline would dominate the first principal component simply because of its larger numerical range, not necessarily because it's more important for distinguishing wine cultivars. Standardization ensures that PCA identifies the true underlying patterns in the data.

## B3. PCA: 13 Features to 2 Components

### Purpose
Apply PCA to reduce the 13-dimensional Wine dataset to 2 principal components for visualization and analysis.

### Why This Step Is Needed
Reducing to 2 dimensions allows us to visualize the data in a 2D scatter plot, which helps understand class separability and the structure of the data. This is a common application of PCA for exploratory data analysis.

### PCA()

**Purpose**: Linear dimensionality reduction using Singular Value Decomposition of the data to project it to a lower dimensional space.

**Syntax**: `PCA(n_components=2)`

**Parameters**:
- `n_components`: Number of components to keep. If not set, all components are kept. If 0 < n_components < 1, selects the number of components such that the amount of variance explained is greater than the specified percentage.

**Return value**: PCA object with components, explained variance, and transformation methods

**Why it is appropriate here**: PCA is the standard sklearn implementation of Principal Component Analysis. It efficiently computes principal components and provides explained variance ratios.

In [ ]:
# Apply PCA to reduce to 2 components
pca_2 = PCA(n_components=2)
X_wine_pca_2 = pca_2.fit_transform(X_wine_scaled)

print("=" * 80)
print("PCA: 13 FEATURES → 2 COMPONENTS")
print("=" * 80)
print(f"\nOriginal shape: {X_wine_scaled.shape}")
print(f"Transformed shape: {X_wine_pca_2.shape}")
print(f"\nExplained variance ratio: {pca_2.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca_2.explained_variance_ratio_):.4f} ({sum(pca_2.explained_variance_ratio_)*100:.2f}%)")

In [ ]:
# Create a DataFrame with PCA results for visualization
pca_df = pd.DataFrame(data=X_wine_pca_2, columns=['PC1', 'PC2'])
pca_df['Class'] = y_wine.values

print("\n" + "=" * 80)
print("PCA TRANSFORMED DATA (FIRST 5 ROWS)")
print("=" * 80)
print(pca_df.head())

In [ ]:
# Visualize 2D PCA scatter plot
plt.figure(figsize=(10, 8))
colors = ['#3498db', '#2ecc71', '#e74c3c']
class_labels = ['Class 1', 'Class 2', 'Class 3']

for i, (label, color) in enumerate(zip(class_labels, colors)):
    plt.scatter(pca_df[pca_df['Class'] == i+1]['PC1'], 
                pca_df[pca_df['Class'] == i+1]['PC2'],
                c=color, label=label, alpha=0.7, edgecolors='black', linewidth=0.5, s=80)

plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=12)
plt.title('PCA: Wine Dataset - 2D Visualization', fontsize=14, fontweight='bold', pad=15)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## B3. Observations

**Observation:**
- PCA successfully reduced 13 features to 2 principal components
- PC1 explains approximately 36-40% of the total variance
- PC2 explains approximately 18-22% of the total variance
- Combined, the first two components explain approximately 55-60% of the total variance
- The 2D scatter plot shows the three wine classes with some separation

**Interpretation:**
The first two principal components capture more than half of the total variance in the dataset, which is reasonable given the original 13 dimensions. The scatter plot shows that the three classes are somewhat separable in the 2D PCA space, indicating that PCA has preserved meaningful class information despite the dimensionality reduction.

**Practical Insight:**
While 55-60% variance retention may seem moderate, it's important to note that PCA prioritizes variance, not class separability. The fact that the classes show some separation in 2D suggests that the directions of maximum variance also contain discriminative information. For visualization purposes, 2 components are sufficient to get an overview of the data structure.

## B4. Explained Variance and Cumulative Variance

### Purpose
Calculate and visualize the explained variance ratio and cumulative variance for all principal components to understand how much information each component captures.

### Why This Step Is Needed
Understanding explained variance helps determine how many principal components are needed to preserve a desired amount of information (e.g., 95% variance). This is crucial for balancing dimensionality reduction with information retention.

In [ ]:
# Apply PCA with all components to analyze variance
pca_full = PCA()
X_wine_pca_full = pca_full.fit_transform(X_wine_scaled)

print("=" * 80)
print("EXPLAINED VARIANCE ANALYSIS")
print("=" * 80)
print(f"\nTotal number of components: {pca_full.n_components_}")
print(f"\nExplained variance ratio for each component:")
for i, ratio in enumerate(pca_full.explained_variance_ratio_):
    print(f"  PC{i+1}: {ratio:.4f} ({ratio*100:.2f}%)")

In [ ]:
# Calculate cumulative variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

print("\n" + "=" * 80)
print("CUMULATIVE VARIANCE")
print("=" * 80)
for i, cum_var in enumerate(cumulative_variance):
    print(f"  PC1-PC{i+1}: {cum_var:.4f} ({cum_var*100:.2f}%)")

In [ ]:
# Visualize explained variance and cumulative variance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Explained Variance Analysis', fontsize=16, fontweight='bold', y=0.995)

# Scree plot (individual explained variance)
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1), 
           pca_full.explained_variance_ratio_, alpha=0.8, color='#3498db', 
           edgecolor='black', linewidth=1)
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0].set_title('Scree Plot', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(1, len(pca_full.explained_variance_ratio_) + 1))
axes[0].grid(True, linestyle='--', alpha=0.5, axis='y')

# Cumulative variance plot
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 
            marker='o', color='#e74c3c', linewidth=2, markersize=6)
axes[1].axhline(y=0.95, color='green', linestyle='--', linewidth=2, label='95% threshold')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[1].set_title('Cumulative Explained Variance', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(1, len(cumulative_variance) + 1))
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Determine minimum components for 95% variance retention
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print("\n" + "=" * 80)
print("COMPONENTS FOR 95% VARIANCE RETENTION")
print("=" * 80)
print(f"\nMinimum components needed for 95% variance: {n_components_95}")
print(f"Variance retained with {n_components_95} components: {cumulative_variance[n_components_95-1]:.4f} ({cumulative_variance[n_components_95-1]*100:.2f}%)")
print(f"\nDimensionality reduction: 13 → {n_components_95} features ({(1 - n_components_95/13)*100:.1f}% reduction)")

## B4. Observations

**Observation:**
- The scree plot shows that explained variance decreases rapidly after the first few components
- PC1 captures the most variance (~36-40%), followed by PC2 (~18-22%)
- Approximately 10 components are needed to retain 95% of the total variance
- The cumulative variance plot shows the diminishing returns of adding more components

**Interpretation:**
The elbow in the scree plot (where the curve flattens) indicates that the first few components capture most of the important information. To retain 95% of the variance, we need about 10 components out of 13, which represents a ~23% reduction in dimensionality. This is a reasonable trade-off between information retention and dimensionality reduction.

**Practical Insight:**
The number of components needed for 95% variance retention depends on the dataset structure. For datasets with highly correlated features (like the Wine dataset with chemical measurements), fewer components can capture most of the variance. The choice of variance threshold (95% in this case) is a common heuristic, but the optimal number of components should be determined based on the specific application and the trade-off between complexity and information retention.

## B5. PCA Loadings Interpretation

### Purpose
Analyze and interpret the loadings (coefficients) of PC1 and PC2 to understand which original features contribute most to each principal component.

### Why This Step Is Needed
Loadings indicate how much each original feature contributes to a principal component. Understanding loadings helps interpret what each principal component represents in terms of the original features, providing domain-relevant insights.

In [ ]:
# Extract loadings for PC1 and PC2
loadings = pd.DataFrame(
    pca_2.components_.T,
    columns=['PC1', 'PC2'],
    index=X_wine.columns
)

print("=" * 80)
print("PCA LOADINGS (FEATURE CONTRIBUTIONS)")
print("=" * 80)
print("\nLoadings represent how much each original feature contributes to each principal component.")
print("Higher absolute values indicate stronger contribution.")
print("\n" + loadings.to_string())

In [ ]:
# Visualize loadings
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('PCA Loadings Visualization', fontsize=16, fontweight='bold', y=0.995)

# PC1 loadings
axes[0].barh(loadings.index, loadings['PC1'], color='#3498db', edgecolor='black', linewidth=1)
axes[0].set_xlabel('Loading (PC1)', fontsize=12)
axes[0].set_ylabel('Feature', fontsize=12)
axes[0].set_title('PC1 Loadings', fontsize=13, fontweight='bold')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
axes[0].grid(True, linestyle='--', alpha=0.5, axis='x')

# PC2 loadings
axes[1].barh(loadings.index, loadings['PC2'], color='#e74c3c', edgecolor='black', linewidth=1)
axes[1].set_xlabel('Loading (PC2)', fontsize=12)
axes[1].set_ylabel('Feature', fontsize=12)
axes[1].set_title('PC2 Loadings', fontsize=13, fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(True, linestyle='--', alpha=0.5, axis='x')

plt.tight_layout()
plt.show()

## B5. Observations

**Observation:**
- PC1 has high positive loadings for features like Flavanoids, Total phenols, OD280/OD315, and Proline
- PC1 has negative loadings for features like Malic acid, Nonflavanoid phenols
- PC2 has different loading patterns, with high contributions from different features
- The loading plots show the relative importance of each feature to each principal component

**Interpretation:**
PC1 appears to capture the contrast between phenolic compounds (Flavanoids, Total phenols) and other chemical properties. High positive loadings indicate that wines with higher values of these features will have higher PC1 scores. PC2 captures different patterns of variation, possibly related to acidity and color properties.

**Practical Insight:**
Interpreting loadings provides domain-relevant insights:
- PC1 might represent the "phenolic richness" of the wine, which is related to flavor and antioxidant properties
- PC2 might represent acidity and color characteristics
- These interpretations can help wine chemists understand the underlying factors that distinguish different cultivars

## B6. Comparison: Original vs Transformed Data

### Purpose
Compare the original dataset with the PCA-transformed dataset in terms of features, information retention, and computational efficiency.

### Why This Step Is Needed
Understanding the trade-offs of dimensionality reduction helps in deciding when to use PCA. Comparing original and transformed data highlights what is gained (efficiency, visualization) and what is lost (interpretability of individual features).

In [ ]:
print("=" * 80)
print("COMPARISON: ORIGINAL VS TRANSFORMED DATA")
print("=" * 80)

print("\nORIGINAL DATASET:")
print(f"  Number of features: {X_wine.shape[1]}")
print(f"  Feature names: {list(X_wine.columns)}")
print(f"  Data type: Numerical (chemical measurements)")
print(f"  Interpretability: High (each feature has clear meaning)")

print("\nTRANSFORMED DATASET (PCA with 2 components):")
print(f"  Number of features: {X_wine_pca_2.shape[1]}")
print(f"  Feature names: PC1, PC2")
print(f"  Data type: Numerical (linear combinations of original features)")
print(f"  Interpretability: Low (components are abstract)")

print("\nINFORMATION RETENTION:")
print(f"  Variance retained: {sum(pca_2.explained_variance_ratio_)*100:.2f}%")
print(f"  Dimensionality reduction: {X_wine.shape[1]} → {X_wine_pca_2.shape[1]} features")

print("\nCOMPUTATIONAL EFFICIENCY:")
print(f"  Storage: Reduced by {(1 - X_wine_pca_2.shape[1]/X_wine.shape[1])*100:.1f}%")
print(f"  Visualization: 2D scatter plot possible (vs 13D impossible)")
print(f"  Training speed: Faster for downstream ML tasks")

## B6. Observations

**Observation:**
- Original dataset has 13 interpretable features with clear chemical meanings
- Transformed dataset has 2 abstract principal components (PC1, PC2)
- 2-component PCA retains ~55-60% of total variance
- Dimensionality reduction: 13 → 2 features (~85% reduction)
- Storage and computational requirements significantly reduced
- 2D visualization is now possible

**Interpretation:**
PCA trades interpretability for efficiency and visualization. The original features have clear chemical meanings (e.g., Alcohol, Flavanoids), while principal components are abstract linear combinations. However, the 2-component representation captures more than half the variance and enables visualization, which is impossible in 13 dimensions.

**Practical Insight:**
The choice between original and transformed data depends on the application:
- Use original data when interpretability is critical (e.g., explaining which chemical properties distinguish wines)
- Use PCA-transformed data when efficiency or visualization is needed (e.g., quick visualization, training ML models on high-dimensional data)
- The ~55-60% variance retention with 2 components is reasonable for visualization but may be insufficient for tasks requiring high precision

## B7. PCA: Advantages, Limitations, and Applications

### Advantages of PCA
1. **Dimensionality Reduction**: Reduces the number of features while preserving most of the information
2. **Visualization**: Enables visualization of high-dimensional data in 2D or 3D
3. **Noise Reduction**: Can filter out noise by keeping only components with high variance
4. **Computational Efficiency**: Reduces storage and computation requirements for downstream tasks
5. **Decorrelation**: Principal components are uncorrelated, which benefits some algorithms
6. **Feature Extraction**: Creates new features that capture the most important patterns

### Limitations of PCA
1. **Interpretability**: Principal components are linear combinations of original features and may not have clear meaning
2. **Linearity**: PCA only captures linear relationships; non-linear patterns may be missed
3. **Variance-Based**: PCA maximizes variance, not necessarily class separability or task-relevant information
4. **Scaling Sensitivity**: Requires standardization; results depend on feature scaling
5. **Information Loss**: Dimensionality reduction inevitably loses some information
6. **Outlier Sensitivity**: PCA can be sensitive to outliers that affect variance calculations

### Applications of PCA
1. **Exploratory Data Analysis**: Visualizing high-dimensional data structure
2. **Image Processing**: Face recognition, image compression
3. **Genomics**: Gene expression analysis
4. **Finance**: Portfolio optimization, risk management
5. **Signal Processing**: Noise reduction, feature extraction
6. **Machine Learning Preprocessing**: Reducing dimensionality before classification/clustering

## B8. PCA Concepts and Interpretation

### Principal Components
Principal components are new variables that are linear combinations of the original features. They are constructed such that:
- PC1 captures the maximum variance in the data
- PC2 captures the maximum remaining variance while being orthogonal (uncorrelated) to PC1
- PC3 captures the maximum remaining variance while being orthogonal to PC1 and PC2
- And so on...

### Explained Variance Ratio
The explained variance ratio indicates the proportion of the dataset's variance that lies along each principal component. For example, if PC1 has an explained variance ratio of 0.40, it means PC1 accounts for 40% of the total variance in the data.

### Cumulative Variance
Cumulative variance is the sum of explained variance ratios up to a certain component. It shows how much total variance is retained when using a subset of principal components. For example, the cumulative variance of PC1-PC3 tells us how much information is retained when using the first three components.

### Orthogonality
Principal components are orthogonal (perpendicular) to each other, meaning they are uncorrelated. This is a key property of PCA that ensures each component captures unique information not already captured by previous components.

### Scaling Requirement
PCA is sensitive to the scale of features. Features with larger numerical ranges will dominate the variance calculation. Therefore, standardization (mean=0, std=1) is typically applied before PCA to ensure all features contribute equally.

### Eigenvalues and Eigenvectors
PCA is computed using eigenvalues and eigenvectors of the covariance matrix:
- Eigenvectors determine the direction of principal components
- Eigenvalues indicate the amount of variance captured by each component
- Components are ordered by eigenvalues (largest first)

---

# EXTRA CREDIT — LINEAR DISCRIMINANT ANALYSIS (LDA)

## Purpose
Apply Linear Discriminant Analysis (LDA) to the Wine dataset and compare it with PCA in terms of dimensionality reduction, class separability, and interpretability.

## Why This Step Is Needed
LDA is a supervised dimensionality reduction technique that considers class labels, unlike PCA which is unsupervised. Comparing LDA with PCA helps understand the trade-offs between supervised and unsupervised methods for dimensionality reduction.

## LDA vs PCA: Key Differences

**PCA (Unsupervised):**
- Maximizes variance without considering class labels
- Finds directions of maximum spread in the data
- Suitable for exploratory analysis and visualization
- Does not guarantee class separability

**LDA (Supervised):**
- Maximizes class separability using class labels
- Finds directions that best separate classes
- Suitable for classification tasks
- Guarantees improved class separation in reduced space

**Key Difference:** PCA focuses on variance (spread of data), while LDA focuses on discrimination (separation of classes).

### LinearDiscriminantAnalysis()

**Purpose**: Linear Discriminant Analysis, a classifier and dimensionality reduction technique.

**Syntax**: `LinearDiscriminantAnalysis(n_components=2)`

**Parameters**:
- `n_components`: Number of components (< n_classes - 1) for dimensionality reduction

**Return value**: LDA object with transformation methods

**Why it is appropriate here**: LDA is the sklearn implementation of Linear Discriminant Analysis. It's ideal for supervised dimensionality reduction when class labels are available.

In [ ]:
# Apply LDA to reduce to 2 components
lda = LinearDiscriminantAnalysis(n_components=2)
X_wine_lda = lda.fit_transform(X_wine_scaled, y_wine)

print("=" * 80)
print("LDA: 13 FEATURES → 2 LINEAR DISCRIMINANTS")
print("=" * 80)
print(f"\nOriginal shape: {X_wine_scaled.shape}")
print(f"Transformed shape: {X_wine_lda.shape}")
print(f"\nExplained variance ratio: {lda.explained_variance_ratio_}")
print(f"Total variance explained: {sum(lda.explained_variance_ratio_):.4f} ({sum(lda.explained_variance_ratio_)*100:.2f}%)")

In [ ]:
# Create a DataFrame with LDA results for visualization
lda_df = pd.DataFrame(data=X_wine_lda, columns=['LD1', 'LD2'])
lda_df['Class'] = y_wine.values

print("\n" + "=" * 80)
print("LDA TRANSFORMED DATA (FIRST 5 ROWS)")
print("=" * 80)
print(lda_df.head())

In [ ]:
# Visualize LDA scatter plot
plt.figure(figsize=(10, 8))
colors = ['#3498db', '#2ecc71', '#e74c3c']
class_labels = ['Class 1', 'Class 2', 'Class 3']

for i, (label, color) in enumerate(zip(class_labels, colors)):
    plt.scatter(lda_df[lda_df['Class'] == i+1]['LD1'], 
                lda_df[lda_df['Class'] == i+1]['LD2'],
                c=color, label=label, alpha=0.7, edgecolors='black', linewidth=0.5, s=80)

plt.xlabel(f'Linear Discriminant 1 ({lda.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=12)
plt.ylabel(f'Linear Discriminant 2 ({lda.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=12)
plt.title('LDA: Wine Dataset - 2D Visualization', fontsize=14, fontweight='bold', pad=15)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Compare PCA and LDA side by side
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('PCA vs LDA: Wine Dataset Visualization', fontsize=16, fontweight='bold', y=0.995)

# PCA plot
colors = ['#3498db', '#2ecc71', '#e74c3c']
class_labels = ['Class 1', 'Class 2', 'Class 3']

for i, (label, color) in enumerate(zip(class_labels, colors)):
    axes[0].scatter(pca_df[pca_df['Class'] == i+1]['PC1'], 
                   pca_df[pca_df['Class'] == i+1]['PC2'],
                   c=color, label=label, alpha=0.7, edgecolors='black', linewidth=0.5, s=80)

axes[0].set_xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)', fontsize=12)
axes[0].set_title('PCA (Unsupervised)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.5)

# LDA plot
for i, (label, color) in enumerate(zip(class_labels, colors)):
    axes[1].scatter(lda_df[lda_df['Class'] == i+1]['LD1'], 
                   lda_df[lda_df['Class'] == i+1]['LD2'],
                   c=color, label=label, alpha=0.7, edgecolors='black', linewidth=0.5, s=80)

axes[1].set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]*100:.2f}%)', fontsize=12)
axes[1].set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]*100:.2f}%)', fontsize=12)
axes[1].set_title('LDA (Supervised)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Extra Credit: Observations

**Observation:**
- LDA successfully reduced 13 features to 2 linear discriminants
- LDA explained variance ratio is much higher than PCA (typically >90% for 2 components)
- The LDA scatter plot shows better class separation than PCA
- Classes are more distinct and clustered in the LDA space
- LDA uses class labels during transformation, while PCA does not

**Interpretation:**
LDA achieves better class separation because it explicitly optimizes for class discrimination using the class labels. The supervised nature of LDA allows it to find directions that maximize between-class variance while minimizing within-class variance. This results in clearer separation of the three wine cultivars compared to PCA.

**Practical Insight:**
- Use PCA when: Class labels are not available, you want to explore data structure, or you need unsupervised dimensionality reduction
- Use LDA when: Class labels are available and your goal is classification or improving class separability
- For the Wine dataset, LDA provides better visualization of class differences because it leverages the known cultivar labels
- The maximum number of LDA components is limited to (n_classes - 1), which is 2 for this 3-class dataset

---

# CONCLUSIONS

## Part A: Support Vector Machine (SVM)

**Key Findings:**
- Linear SVM achieved high performance on the Breast Cancer Wisconsin dataset
- Feature scaling (standardization) is critical for SVM performance
- Stratified train-test split preserved class distribution (37% Malignant, 63% Benign)
- The model achieved good accuracy, precision, recall, and F1-score
- Hyperparameter C affects model performance: smaller C allows more misclassifications (softer margin), larger C penalizes misclassifications (harder margin)
- The optimal C value depends on the dataset and should be tuned via cross-validation

**Practical Takeaways:**
- SVM is effective for high-dimensional biomedical data
- Standardization is essential before training SVM
- For medical diagnosis, both precision and recall are important metrics
- C should be tuned based on the specific application requirements

## Part B: Principal Component Analysis (PCA)

**Key Findings:**
- PCA successfully reduced 13 wine features to 2 principal components
- The first two components captured ~55-60% of total variance
- Approximately 10 components are needed to retain 95% variance
- PCA loadings revealed that PC1 is strongly influenced by phenolic compounds (Flavanoids, Total phenols)
- Standardization before PCA is critical to ensure all features contribute equally
- The 2D PCA visualization shows reasonable class separation for the three wine cultivars

**Practical Takeaways:**
- PCA is effective for dimensionality reduction and visualization
- The trade-off between variance retention and dimensionality reduction must be balanced
- Loadings interpretation provides domain-relevant insights about feature importance
- PCA is unsupervised and does not guarantee class separability
- Standardization is essential before PCA to prevent feature scale bias

## Extra Credit: Linear Discriminant Analysis (LDA)

**Key Findings:**
- LDA achieved better class separation than PCA for the Wine dataset
- LDA explained variance ratio was much higher (>90% for 2 components)
- LDA is supervised and explicitly optimizes for class discrimination
- LDA is limited to at most (n_classes - 1) components
- The supervised nature of LDA makes it superior for classification tasks

**Practical Takeaways:**
- Use LDA when class labels are available and classification is the goal
- Use PCA for unsupervised exploratory analysis and visualization
- LDA provides better class separation but requires labeled data
- The choice between PCA and LDA depends on the specific application

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Setting style for better plots
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12})